In [ ]:
!pip install -q sktime psutil

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = '/content/drive/MyDrive/diss-datasets'

import os
os.makedirs(DATA_DIR, exist_ok=True)
print('Currently in DATA_DIR:', os.listdir(DATA_DIR))

In [ ]:
import numpy as np
import pandas as pd
import time
import gc
import psutil
from sklearn.linear_model import RidgeClassifierCV
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support

DATASETS = {
    'CMJ':     {'file': 'CMJ.npy',                 'binary': False},
    'MP8':     {'file': 'MP8.npy',                 'binary': False},
    'MP50':    {'file': 'MP50.npy',                'binary': False},
    'Rowing':  {'file': 'Rowing.npy',              'binary': False},
    'SYNTH':   {'file': 'SYNTH.npy',               'binary': True},
    'CHB-MIT': {'file': 'CHBMIT_crosspatient.npy', 'binary': True},
}

for name, meta in DATASETS.items():
    path = f'{DATA_DIR}/{meta["file"]}'
    print(name, 'found' if os.path.exists(path) else 'MISSING', path)

In [ ]:
def load_dataset(fname):
    d = np.load(f'{DATA_DIR}/{fname}', allow_pickle=True).item()
    X_tr = d['train']['X'].astype(np.float32)
    y_tr = np.array([int(v) for v in d['train']['y']])
    X_te = d['test']['X'].astype(np.float32)
    y_te = np.array([int(v) for v in d['test']['y']])
    return X_tr, y_tr, X_te, y_te

def evaluate(y_true, y_pred, binary):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    out = {'accuracy': acc, 'macro_f1': macro_f1}
    if binary:
        p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', pos_label=1)
        out.update({'pos_precision': p, 'pos_recall': r, 'pos_f1': f1})
    return out

def ram_gb():
    return psutil.virtual_memory().used / 1e9

def free_ram_gb():
    return psutil.virtual_memory().available / 1e9

def to_f32(df):
    return np.ascontiguousarray(df.to_numpy(dtype=np.float32))

FEATURE_BUDGET = 50000
FEATURE_BUDGET_HIGHRISK = 20000
RISK_THRESHOLD = 1_000_000
MINI_FEATS_PER_KERNEL = 1
MULTI_FEATS_PER_KERNEL = 8

def risk_metric(X_tr):
    return X_tr.shape[0] * X_tr.shape[2]

def budget_for(X_tr):
    return FEATURE_BUDGET_HIGHRISK if risk_metric(X_tr) >= RISK_THRESHOLD else FEATURE_BUDGET

def kernels_for(budget, feats_per_kernel, n_channels):
    k = budget // (feats_per_kernel * n_channels)
    k = int(round(k / 84) * 84)
    return max(84, k)

In [ ]:
from sktime.transformations.panel.rocket import MiniRocketMultivariate, MiniRocket

def run_minirocket_multivariate(X_tr, y_tr, X_te, y_te, binary, num_kernels):
    t0 = time.time()
    rocket = MiniRocketMultivariate(num_kernels=num_kernels)
    rocket.fit(X_tr)
    feat_tr = to_f32(rocket.transform(X_tr))
    feat_te = to_f32(rocket.transform(X_te))
    del rocket
    gc.collect()
    clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    clf.fit(feat_tr, y_tr)
    pred = clf.predict(feat_te)
    elapsed = time.time() - t0
    result = evaluate(y_te, pred, binary)
    result['seconds'] = elapsed
    result['num_kernels'] = num_kernels
    result['n_features'] = feat_tr.shape[1]
    del feat_tr, feat_te, clf
    gc.collect()
    return result

def run_minirocket_singlechannel(X_tr, y_tr, X_te, y_te, binary, num_kernels):
    t0 = time.time()
    feats_tr, feats_te = [], []
    n_channels = X_tr.shape[1]
    for ch in range(n_channels):
        r = MiniRocket(num_kernels=num_kernels)
        Xc_tr = np.ascontiguousarray(X_tr[:, ch:ch+1, :])
        Xc_te = np.ascontiguousarray(X_te[:, ch:ch+1, :])
        r.fit(Xc_tr)
        feats_tr.append(to_f32(r.transform(Xc_tr)))
        feats_te.append(to_f32(r.transform(Xc_te)))
        del r, Xc_tr, Xc_te
        gc.collect()
    feat_tr = np.concatenate(feats_tr, axis=1)
    feat_te = np.concatenate(feats_te, axis=1)
    del feats_tr, feats_te
    gc.collect()
    clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    clf.fit(feat_tr, y_tr)
    pred = clf.predict(feat_te)
    elapsed = time.time() - t0
    result = evaluate(y_te, pred, binary)
    result['seconds'] = elapsed
    result['num_kernels'] = num_kernels
    result['n_features'] = feat_tr.shape[1]
    del feat_tr, feat_te, clf
    gc.collect()
    return result

In [ ]:
from sktime.transformations.panel.rocket import MultiRocketMultivariate, MultiRocket

def run_multirocket_multivariate(X_tr, y_tr, X_te, y_te, binary, num_kernels):
    t0 = time.time()
    rocket = MultiRocketMultivariate(num_kernels=num_kernels)
    X_tr64 = np.ascontiguousarray(X_tr, dtype=np.float64)
    rocket.fit(X_tr64)
    feat_tr = to_f32(rocket.transform(X_tr64))
    del X_tr64
    gc.collect()
    X_te64 = np.ascontiguousarray(X_te, dtype=np.float64)
    feat_te = to_f32(rocket.transform(X_te64))
    del X_te64, rocket
    gc.collect()
    clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    clf.fit(feat_tr, y_tr)
    pred = clf.predict(feat_te)
    elapsed = time.time() - t0
    result = evaluate(y_te, pred, binary)
    result['seconds'] = elapsed
    result['num_kernels'] = num_kernels
    result['n_features'] = feat_tr.shape[1]
    del feat_tr, feat_te, clf
    gc.collect()
    return result

def run_multirocket_singlechannel(X_tr, y_tr, X_te, y_te, binary, num_kernels):
    t0 = time.time()
    feats_tr, feats_te = [], []
    n_channels = X_tr.shape[1]
    for ch in range(n_channels):
        r = MultiRocket(num_kernels=num_kernels)
        Xc_tr = np.ascontiguousarray(X_tr[:, ch:ch+1, :], dtype=np.float64)
        Xc_te = np.ascontiguousarray(X_te[:, ch:ch+1, :], dtype=np.float64)
        r.fit(Xc_tr)
        feats_tr.append(to_f32(r.transform(Xc_tr)))
        feats_te.append(to_f32(r.transform(Xc_te)))
        del r, Xc_tr, Xc_te
        gc.collect()
    feat_tr = np.concatenate(feats_tr, axis=1)
    feat_te = np.concatenate(feats_te, axis=1)
    del feats_tr, feats_te
    gc.collect()
    clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
    clf.fit(feat_tr, y_tr)
    pred = clf.predict(feat_te)
    elapsed = time.time() - t0
    result = evaluate(y_te, pred, binary)
    result['seconds'] = elapsed
    result['num_kernels'] = num_kernels
    result['n_features'] = feat_tr.shape[1]
    del feat_tr, feat_te, clf
    gc.collect()
    return result

In [ ]:
RESULTS_PATH = f'{DATA_DIR}/rocket_all_results_v2.csv'

if os.path.exists(RESULTS_PATH):
    done_df = pd.read_csv(RESULTS_PATH)
    valid = done_df[done_df['accuracy'].notna()]
    done_keys = set(zip(valid['dataset'], valid['method'], valid['variant']))
    print(f'Resuming, {len(done_keys)} valid results already saved')
else:
    done_df = pd.DataFrame()
    done_keys = set()

def save_result(r):
    global done_df, done_keys
    done_df = pd.concat([done_df, pd.DataFrame([r])], ignore_index=True)
    done_df.to_csv(RESULTS_PATH, index=False)
    done_keys.add((r['dataset'], r['method'], r['variant']))

JOBS = [
    ('MiniRocket',  'multivariate',   run_minirocket_multivariate,   MINI_FEATS_PER_KERNEL,  False),
    ('MiniRocket',  'single-channel', run_minirocket_singlechannel,  MINI_FEATS_PER_KERNEL,  True),
    ('MultiRocket', 'multivariate',   run_multirocket_multivariate,  MULTI_FEATS_PER_KERNEL, False),
    ('MultiRocket', 'single-channel', run_multirocket_singlechannel, MULTI_FEATS_PER_KERNEL, True),
]

for name, meta in DATASETS.items():
    print(f'\n=== {name} ===')
    X_tr, y_tr, X_te, y_te = load_dataset(meta['file'])
    n_channels = X_tr.shape[1]
    risk = risk_metric(X_tr)
    budget = budget_for(X_tr)
    print(f'shape train {X_tr.shape}  test {X_te.shape}  channels {n_channels}  risk {risk}  feature budget {budget}')

    for method, variant, fn, fpk, per_channel in JOBS:
        if (name, method, variant) in done_keys:
            print(f'{method} {variant} skipped, already done')
            continue
        nk = kernels_for(budget, fpk, n_channels if per_channel else 1)
        est_gb = X_tr.shape[0] * budget * 4 / 1e9 * 3
        print(f'{method} {variant} running, num_kernels {nk}, est peak {est_gb:.1f} GB, free {free_ram_gb():.1f} GB')
        r = fn(X_tr, y_tr, X_te, y_te, meta['binary'], nk)
        r.update({'dataset': name, 'method': method, 'variant': variant,
                  'n_channels': n_channels, 'risk': risk, 'feature_budget': budget})
        print(f"{method} {variant} acc={r['accuracy']:.4f} macroF1={r['macro_f1']:.4f} "
              f"feats={r['n_features']} ({r['seconds']:.1f}s) RAM now {ram_gb():.1f} GB")
        save_result(r)
        gc.collect()

    del X_tr, y_tr, X_te, y_te
    gc.collect()

print('\nall done')

In [ ]:
df = pd.read_csv(RESULTS_PATH)
pd.set_option('display.width', 160)
print(df[['dataset', 'method', 'variant', 'n_channels', 'num_kernels', 'n_features',
          'accuracy', 'macro_f1', 'seconds']].to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt

order = ['CMJ', 'MP8', 'MP50', 'Rowing', 'SYNTH', 'CHB-MIT']

for method in ['MiniRocket', 'MultiRocket']:
    sub = df[df['method'] == method]
    pivot = sub.pivot(index='dataset', columns='variant', values='macro_f1').reindex(order)

    fig, ax = plt.subplots(figsize=(9, 5))
    x = np.arange(len(pivot))
    width = 0.35
    ax.bar(x - width/2, pivot['multivariate'], width, label='Multivariate (channel-aware)')
    ax.bar(x + width/2, pivot['single-channel'], width, label='Single-channel (channel-independent)')
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index)
    ax.set_ylabel('Macro F1')
    ax.set_title(f'{method}: channel-independence penalty across domains')
    ax.legend()
    plt.tight_layout()
    plt.savefig(f'{DATA_DIR}/rocket_channel_penalty_{method}.png', dpi=150)
    plt.show()

penalty = df.pivot_table(index=['dataset', 'method'], columns='variant', values='macro_f1')
penalty['penalty'] = penalty['multivariate'] - penalty['single-channel']
print(penalty.round(4).to_string())

In [ ]:
chbmit_scores = {
    'Majority baseline': 0.6667,
    'Correlation features': 0.7318,
    'Band power features': 0.8218,
    'Band power + correlation': 0.8678,
    'MantisV2 (frozen probe)': 0.7950,
    'Chronos-2 (frozen probe)': 0.8199,
}
chbmit_rocket = df[df['dataset'] == 'CHB-MIT'].set_index(['method', 'variant'])['accuracy']
for (method, variant), val in chbmit_rocket.items():
    chbmit_scores[f'{method} ({variant})'] = val

items = sorted(chbmit_scores.items(), key=lambda kv: kv[1])
labels, vals = zip(*items)

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(labels, vals)
ax.set_xlabel('Accuracy')
ax.set_title('CHB-MIT: all methods compared')
ax.set_xlim(0.6, 0.9)
plt.tight_layout()
plt.savefig(f'{DATA_DIR}/chbmit_all_methods.png', dpi=150)
plt.show()

In [ ]:
import json
records = df.to_dict(orient='records')
with open(f'{DATA_DIR}/rocket_summary.json', 'w') as f:
    json.dump(records, f, indent=2, default=float)
print('saved', len(records), 'records')